In [ ]:
! pip install statsmodels

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from jupyter_dash import JupyterDash
from dash import Dash, dcc, html, Input, Output
import statsmodels


git remote add origin https://github.com/bacazachary92/NFL-Fantasy-Dashboard-Project
git push -u origin main



### Exploratory Data Analysis

In [163]:
nfl = pd.read_csv("fantasy_data.csv")

#https://www.kaggle.com/datasets/heefjones/nfl-fantasy-data-1970-2024/data

In [164]:

print("Head \n ", nfl.head(10))
#Potential issue with duplicates -- will see how it affects graphs
print("______________________________________________________________________")

print("Description\n", nfl.describe().T)
print("______________________________________________________________________")

print("Missing Values\n", nfl.isna().sum())

print("______________________________________________________________________")

#Players will change teams mid season -- trades. This will be handled in a later cell 
print("Duplicates\n", nfl.duplicated())

print("______________________________________________________________________")


multiple_teams = nfl.groupby('Player')['Tm'].nunique()
print(multiple_teams[multiple_teams > 2].sort_values(ascending=False).head(20))


print("______________________________________________________________________")

print("Types\n", nfl.dtypes)

print("______________________________________________________________________")
print("Note: Two players named Adrian Peterson exist in dataset (CHI 2002, MIN 2007)\n CHI version is obscure and not relevant for fantasy football purposes\n Known edge case - acceptable for this use case")
## Graphs created a zigzag for people with the same name. Not just traded mid season. 

ap = nfl[nfl['Player'] == 'Adrian Peterson']
print(ap[['Player', 'Year', 'Tm', 'Points_half-ppr']])

Head 
                Player   Tm Pos  Age   G  GS  Pass_Cmp  Pass_Att  Pass_Yds  \
0  Israel Abanikanda  NYJ  RB   21   6   0         0         0         0   
1   Jared Abbrederis  GNB  WR   25  10   0         0         0         0   
2   Jared Abbrederis  GNB  WR   26   5   0         0         0         0   
3   Jared Abbrederis  DET  WR   27   7   0         0         0         0   
4     Ameer Abdullah  DET  RB   22  16   9         0         0         0   
5     Ameer Abdullah  DET  RB   23   2   2         0         0         0   
6     Ameer Abdullah  DET  RB   24  14  11         0         0         0   
7     Ameer Abdullah  2TM  RB   25  10   0         0         0         0   
8     Ameer Abdullah  MIN  RB   26  16   0         0         0         0   
9     Ameer Abdullah  MIN  RB   27  16   0         0         0         0   

   Pass_TD  ...  PointsOvrRank_half-ppr  PointsPosRank_half-ppr  \
0        0  ...                     400                      99   
1        0  ...      

In [ ]:
nfl.describe().T

In [ ]:
nfl.columns

In [165]:

#All the traded players have 2TM as a team. 
traded = nfl[nfl['Tm'] == '2TM']

#Not traded 
not_traded = nfl[nfl['Tm'] != '2TM']

#Combine the player traded with the year
traded_pairs = set(zip(traded["Player"], traded ["Year"]))



#This is a complicated line: Let's go through the logic
#Lambda row receives each row as input to check if that player was traded.
#pply checks every row, axis = 1 goes row by row
#~ flips the boolean which logicall says keep the rows that are not in the traded_player list
not_traded_clean = not_traded[~not_traded.apply(
    lambda row: (row["Player"], row["Year"]) in traded_pairs, axis =1

)]

nfl_clean = pd.concat([traded, not_traded_clean]).sort_values(["Player", "Year"]).reset_index(drop = True)


### Building Plots for Dash

#### Skill players over time -- measured by their age. Does father time always win?

In [166]:
#DEFINE PLAYERS -- DROP DOWN?
rb = ["Ezekiel Elliot", "Jordan Howard", "DeMarco Murray", "Jay Ajayi", "Le'Veon Bell"]

#filter players by position, sort by age.
filtered_nfl = nfl_clean[nfl_clean["Player"].isin(rb)]
filtered_nfl = filtered_nfl.sort_values('Age')

#Figure with columns selelected by position played. 
fig = px.line(filtered_nfl, x = "Age", y = "PPG_half-ppr", color = "Player", hover_data= {"Tm":True, "Year":True, "Touches_per_game": True, 'Rush_TD': True, "Rush_Yds": True,"Rec_TD": True})
fig.show()

In [167]:
wr = ["Larry Fitzgerald","Odell Beckham Jr.", "Julian Edelman", "Antonio Brown"]  # WR

filtered_nfl = nfl[nfl["Player"].isin(wr)]
filtered_nfl = filtered_nfl.sort_values('Age')

fig = px.line(filtered_nfl, x = "Age", y = "PPG_half-ppr", color = "Player", hover_data= {"Tm":True , "Rec_Yds_per_game": True, "Year":True,"Rec_Tgt_per_game": True, "Rec_TD_per_game": True })
fig.show()

In [168]:
qb =["Drew Brees", "Peyton Manning", "Aaron Rodgers", "Tom Brady"] 


filtered_nfl = nfl[nfl["Player"].isin(qb)]
filtered_nfl = filtered_nfl.sort_values('Age')

fig = px.line(filtered_nfl, x = "Age", y = "PPG_half-ppr", color = "Player", hover_data= {"Year":True, "Tm":True , "Pass_Att": True, "Pass_Cmp%": True, "Pass_TD_per_game": True,"Pass_Int_per_game": True })
fig.show()

In [ ]:
te =["Zach Ertz", "Travis Kelce", "Sam LaPorta", "David Njoku", "Trey McBride"] 


filtered_nfl = nfl[nfl["Player"].isin(te)]
filtered_nfl = filtered_nfl.sort_values('Age')

fig = px.line(filtered_nfl, x = "Age", y = "PPG_half-ppr", color = "Player", hover_data= {"Tm":True,"Year":True , "Rec_Yds_per_game": True, "Rec_Tgt_per_game": True, "Rec_TD_per_game": True })
fig.show()

## Defining a Function to bring all the plots above together. 

In [184]:
"""
get_hover_data
Input: position the player plays
Output: The columns included in the hover. 

Players by position
Input: A list of players, and position
Output: Line graph showing points by age 
"""
players = ["Brock Bowers", "Zach Ertz", "Travis Kelce", "Sam LaPorta", "David Njoku", "Trey McBride", "Drew Brees", "Peyton Manning", "Aaron Rodgers", "Tom Brady","Larry Fitzgerald","Odell Beckham Jr.", "Julian Edelman", "Antonio Brown","Ezekiel Elliot", "Jordan Howard", "DeMarco Murray", "Jay Ajayi", "Le'Veon Bell" ] 

def get_hover_columns(selected_position):
    if selected_position == "QB":
        return {"Year":True, "Tm":True , "Pass_Att": True, "Pass_Cmp%": True, "Pass_TD_per_game": True,"Pass_Int_per_game": True }
    elif selected_position == "WR" or selected_position == "TE":
        return {"Tm":True , "Rec_Yds_per_game": True, "Year":True,"Rec_Tgt_per_game": True, "Rec_TD_per_game": True }
    elif selected_position == "RB":
        return {"Tm":True, "Year":True, "Touches_per_game": True, 'Rush_TD': True, "Rush_Yds": True,"Rec_TD": True}
    
def players_by_position(selected_players, selected_position, selected_stat):
    filtered_nfl = nfl[nfl["Player"].isin(selected_players)]
    filtered_nfl = filtered_nfl[filtered_nfl["Pos"] == selected_position]
    hover_cols = get_hover_columns(selected_position)
    
    fig = px.line(filtered_nfl, x = "Age", y = selected_stat, color = "Player", hover_data= hover_cols)
    fig.update_layout(title = f"Career Trajectory for {selected_players} by Age ")
    
    return fig



In [179]:
## Another plot to add
#See how players stack up to their peers in a given year on a given stat

filtered = nfl[nfl['Pos'] == 'RB']  # one position
filtered = filtered[filtered['Year'] == 2008]  # one year
top10 = filtered.nlargest(20, 'Scrim_TD')

fig = px.bar(top10, x ='Scrim_TD' , y = "Player", orientation='h')
fig.show()



In [185]:
"""
Input: position, year and stat
Output: plot showing the top 20 players at that position in that stat
    """
def top_twenty_players(selected_position, selected_year, selected_stat):
    hover_data = {"Tm": True, "Age": True, "G":True}
    filtered_nfl = nfl_clean[nfl_clean['Pos'] == selected_position] 
    filtered_nfl = filtered_nfl[filtered_nfl['Year'] == selected_year] 
    top = filtered_nfl.nlargest(20, selected_stat)

    fig = px.bar(top, x = selected_stat, y = 'Player', hover_data= hover_data, orientation= "h")
    fig.update_layout(title = f"Top 20 {selected_position} in the League in {selected_year}")
    return fig
    

In [186]:
"""
Function to build a plot for Which teams have the highest scoring fantasy outputs per skill position in a given year. 
Input: Position, year
Output: Plot for highest point totals for skill positions each year
"""
def fantasy_points_by_team(selected_position, selected_year):
    

    #Filter the data frame by postion, and year
    filtered = nfl_clean[nfl_clean['Pos']== selected_position]
    filtered = filtered[filtered['Year'] == selected_year]
    fantasy_by_team = filtered.groupby(["Tm"]).agg(
        Points_half_ppr = ('Points_half-ppr', "mean"), 
        Rec_TD = ("Rec_TD", "mean"),
        Rush_TD = ("Rush_TD", "mean"), 
        Pass_TD = ("Pass_TD", "mean")
        ).reset_index()
    
    #What data shows in the hover
    hover_info = {"Rec_TD": ":.1f", "Rush_TD":":.1f", "Pass_TD": ":.1f" }

    #Create the bar graph
    fig = px.bar(fantasy_by_team, x = "Tm", y = 'Points_half_ppr', orientation = "v", hover_data=hover_info)

    fig.update_layout(title = f"Fantasy Points per team by {selected_position}",
                      xaxis = {'categoryorder': 'mean descending'})

    #Create a reference line for the average fantasy points
    line= nfl_clean["Points_half-ppr"].mean()
    fig.add_hline(y = line, line_width = 2, line_color = "red" )
    return fig


In [187]:
"""
Input: Skill position and year
Output: Plot to find the most efficient players in the league
Players above the line are above the league average
"""

def player_value_scat (selected_position, selected_year, selected_players = None):
    #Filter by pos and year 
    filtered = nfl_clean[nfl_clean['Pos'].copy()== selected_position]
    filtered = filtered[filtered['Year'] == selected_year]

    #Dynamic coloring -- connecting the player selected to th eother graphs
    if selected_players and len(selected_players) >0:
        filtered["Color Status"] = np.where(
            filtered["Player"].isin(selected_players), 
            "Selected Player", 
            "Other Players"
        )
        filtered["Size_Value"] = np.where(
            filtered["Player"].isin(selected_players),
            25,
            10
        )
        #Making the selected player stand out
        color_map = {"Selected Player":'hotpink', 'Other Players': '#1f77b4' }

    else:
        #No selected players, then all are blue
        filtered["Color Status"] = "All Players"
        color_map = {'All Players': '#1f77b4'} # Standard blue

    fig = px.scatter(filtered, x = "Touches_per_game", 
                 y = 'PPG_half-ppr', 
                 color = "Color Status",
                 color_discrete_map= color_map,
                 size = "Size_Value", 
                 size_max= 18,
                 hover_name= "Player", 
                 hover_data=["Tm", "Age"], 
                 trendline= "ols",
                 trendline_color_override="red")
    
    fig.update_layout(title = f"{selected_position} Effeciency vs Volume {selected_players}")
             
    return fig 


## Building the Dashboard

In [189]:

app = Dash()

app.layout = html.Div([
    dcc.Dropdown(
        options = ["QB", "WR", "RB", "TE"], 
        value = "QB", 
        id = "position-dropdown"
    ),
    dcc.Dropdown(
        id = 'player-dropdown', 
        multi = True

    ),
     
#Controls the year selected (Useful for the bargraph)

    dcc.Dropdown(
        options = [{'label': year, "value" : year} for year in sorted(nfl["Year"].unique(), reverse= True)],
        value = 2023,
        id = "year-dropdown"
    ),
    #Changes the type of stats presented
    dcc.RadioItems(
        options = [
            {"label": "Fantasy Points", "value": 'Points_half-ppr'}, 
            {"label": "Avg Fantasy Points", "value": 'PPG_half-ppr'},
            {"label": "Passing TD", "value": 'Pass_TD'}, #Passing TD
            {"label": "Rushing TD", "value": 'Rush_TD'}, #Rushing TD
            {"label": "Receiving TD", "value": 'Rec_TD'}], #Rec TD 
        value = 'Points_half-ppr',
        id = "stat-selector",
        style = {'backgroundColor': 'white'}
),
    dcc.Graph(id = "player-graph"),
    dcc.Graph(id = "top-twenty-graph"),
    dcc.Graph(id = "points-by-team-graph"),
    dcc.Graph(id = "scatter-pos-graph")

]
)

#Callback #1: Updates the players position
@app.callback(
    Output('player-dropdown', 'options'), 
    Input('position-dropdown', 'value')
)

def update_player_options(selected_position):
    if not selected_position:
        return []
    
    selected_players = nfl[nfl["Pos"] == selected_position]
    return sorted(selected_players["Player"].unique())


#Callback #2: Updates the individual player line graph
@app.callback(
    Output("player-graph", "figure"), 
    Input("player-dropdown", "value"), 
    Input("position-dropdown", "value"),
    Input("stat-selector", "value")
)
def update_line_graph(selected_players, selected_position, selected_stat):
    if not selected_players:
        return px.line(title = "Select a skill position from the dropdown above")
    fig = players_by_position(selected_players, selected_position, selected_stat)
    return fig

#Callback #3 Define the inputs for 3 charts
@app.callback(
    Output("top-twenty-graph", "figure"),
    Output("points-by-team-graph", "figure"),
    Output("scatter-pos-graph", "figure"), 
    Input("position-dropdown", "value"),
    Input("year-dropdown", "value"),
    Input("stat-selector", "value"),
    Input("player-dropdown", "value"),
    prevent_initial_call=True 
)

def updated_charts(selected_position, selected_year, selected_stat, selected_players):
    if selected_players is None:
        selected_players = []
        
    if not selected_position or not selected_year:
        blank_chart = px.bar(title = "Select parameters to generate graphs")
        return blank_chart, blank_chart, blank_chart
    
#Generate the three graphs

    fig_top_twenty = top_twenty_players(selected_position, selected_year, selected_stat)
    fig_team_points = fantasy_points_by_team(selected_position, selected_year)
    fig_scatter = player_value_scat(selected_position, selected_year, selected_players)

    return fig_top_twenty, fig_team_points, fig_scatter


if __name__ == '__main__':
    # Setting debug=True opens an interactive error pop-up with dash
    app.run(jupyter_mode="tab", port=505, debug=True)


Dash app running on http://127.0.0.1:505/


<IPython.core.display.Javascript object>

## Other Plots we need to incorporate

In [ ]:
## Which teams have the highest scoring fantasy outputs per skill position in a given year. 
#There are other factors that go into it, but the dataset does not have coaches or strategy in it. 

pos = "QB" #Skill position
year = 2023 #Dropdown

filtered = nfl_clean[nfl_clean['Pos']== pos]
filtered = filtered[filtered['Year'] == year]
fantasy_by_team = filtered.groupby(["Tm"])['Points_half-ppr'].mean().reset_index()

fig = px.bar(fantasy_by_team, x = "Tm", y = 'Points_half-ppr', orientation = "v")
line= nfl_clean["Points_half-ppr"].mean()
fig.add_hline(y = line, line_width = 2, line_color = "red" )
fig.show()

In [ ]:
"""
Function to build a plot for Which teams have the highest scoring fantasy outputs per skill position in a given year. 
Input: Position, year
Output: Plot for highest point totals for skill positions each year
"""
def fantasy_points_by_team(selected_position, selected_year):
    

    #Filter the data frame by postion, and year
    filtered = nfl_clean[nfl_clean['Pos']== selected_position]
    filtered = filtered[filtered['Year'] == selected_year]
    fantasy_by_team = filtered.groupby(["Tm"]).agg(
        Points_half_ppr = ('Points_half-ppr', "mean"), 
        Rec_TD = ("Rec_TD", "mean"),
        Rush_TD = ("Rush_TD", "mean"), 
        Pass_TD = ("Pass_TD", "mean")
        ).reset_index()
    
    #What data shows in the hover
    hover_info = {"Rec_TD": ":.1f", "Rush_TD":":.1f", "Pass_TD": ":.1f" }

    #Create the bar graph
    fig = px.bar(fantasy_by_team, x = "Tm", y = 'Points_half_ppr', orientation = "v", hover_data=hover_info)

    fig.update_layout(xaxis = {'categoryorder': 'mean descending'})

    #Create a reference line for the average fantasy points
    line= nfl_clean["Points_half-ppr"].mean()
    fig.add_hline(y = line, line_width = 2, line_color = "red" )
    fig.show()


In [ ]:
result = fantasy_points_by_team('QB', 1999)
result 


In [ ]:
## The Workhorse Metric

selected_position = "TE" #Skill position
selected_year = 2015 #Dropdown

filtered = nfl_clean[nfl_clean['Pos']== selected_position]
filtered = filtered[filtered['Year'] == selected_year]



hover_info = {"Rec_TD": ":.1f", "Rush_TD":":.1f", "Pass_TD": ":.1f" }


fig = px.scatter(filtered, x = "Touches_per_game", 
                 y = 'PPG_half-ppr', 
                 hover_name= "Player", 
                 hover_data=["Tm", "Age"], 
                 trendline= "ols",
                 trendline_color_override="red")

fig.show()

In [ ]:
def player_value_scat (selected_position, selected_year):
    #Filter by pos and year 
    filtered = nfl_clean[nfl_clean['Pos']== selected_position]
    filtered = filtered[filtered['Year'] == selected_year]

    fig = px.scatter(filtered, x = "Touches_per_game", 
                 y = 'PPG_half-ppr', 
                 hover_name= "Player", 
                 hover_data=["Tm", "Age"], 
                 trendline= "ols",
                 trendline_color_override="red")
    fig.show()


In [ ]:
result = player_value_scat("RB", 2016)
result 